In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [6]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

candidate_path = (
    project_root
    / "data"
    / "interim"
    / "sentinel5p_ch4_po_valley_candidate_2019_2024.nc"
)

candidate_ds = xr.open_dataset(candidate_path)
ch4_candidate = candidate_ds["CH4"]

In [11]:
ch4_stacked = (
    ch4_candidate
    .stack(cell=("y", "x"))
    .dropna(dim="cell", how="all")
)

ch4_df = (
    ch4_stacked
    .to_series()
    .rename("CH4")
    .reset_index()
    .rename(columns={"t": "week"})
    .sort_values(["y", "x", "week"])
    .reset_index(drop=True)
)

In [12]:
ch4_df["cell_id"] = (
    ch4_df
    .groupby(["y", "x"], sort=True)
    .ngroup()
)

In [13]:
print("Rows:", len(ch4_df))
print("Cells:", ch4_df["cell_id"].nunique())
print("Weeks:", ch4_df["week"].nunique())

assert ch4_df["cell_id"].nunique() == 2620
assert ch4_df["week"].nunique() == 314
assert len(ch4_df) == 2620 * 314

Rows: 822680
Cells: 2620
Weeks: 314


In [16]:
ch4_df = (
    ch4_df
    .sort_values(["cell_id", "week"])
    .reset_index(drop=True)
)

In [17]:
cell_groups = ch4_df.groupby(
    "cell_id",
    sort=False,
)

ch4_df["ch4_prev_1w"] = (
    cell_groups["CH4"].shift(1)
)

ch4_df["ch4_prev_2w"] = (
    cell_groups["CH4"].shift(2)
)

ch4_df["ch4_change_1w"] = (
    ch4_df["CH4"]
    - ch4_df["ch4_prev_1w"]
)

ch4_df["target_next_week"] = (
    cell_groups["CH4"].shift(-1)
)

ch4_df["target_week"] = (
    cell_groups["week"].shift(-1)
)

In [18]:
ch4_df["ch4_mean_last_3w"] = (
    cell_groups["CH4"]
    .rolling(
        window=3,
        min_periods=3,
    )
    .mean()
    .reset_index(
        level=0,
        drop=True,
    )
)

ch4_df["ch4_std_last_3w"] = (
    cell_groups["CH4"]
    .rolling(
        window=3,
        min_periods=3,
    )
    .std()
    .reset_index(
        level=0,
        drop=True,
    )
)

In [19]:
week_of_year = (
    ch4_df["week"]
    .dt
    .isocalendar()
    .week
    .astype(int)
)

ch4_df["season_sin"] = np.sin(
    2 * np.pi * week_of_year / 52
)

ch4_df["season_cos"] = np.cos(
    2 * np.pi * week_of_year / 52
)

In [20]:
available_targets = ch4_df[
    ch4_df["target_next_week"].notna()
].copy()

forecast_horizon_days = (
    available_targets["target_week"]
    - available_targets["week"]
).dt.days

print(
    forecast_horizon_days.value_counts()
)

assert forecast_horizon_days.eq(7).all()

7    519725
Name: count, dtype: int64


In [21]:
feature_columns = [
    "CH4",
    "ch4_prev_1w",
    "ch4_prev_2w",
    "ch4_change_1w",
    "ch4_mean_last_3w",
    "ch4_std_last_3w",
    "x",
    "y",
    "season_sin",
    "season_cos",
]

target_column = "target_next_week"

required_columns = (
    feature_columns
    + [target_column]
)

model_df = (
    ch4_df
    .dropna(subset=required_columns)
    .copy()
    .reset_index(drop=True)
)

In [22]:
print("Original rows:", len(ch4_df))
print("Model-ready rows:", len(model_df))

retention_rate = (
    len(model_df) / len(ch4_df)
)

print(
    f"Retention rate: "
    f"{retention_rate:.2%}"
)

print(
    "Model-ready cells:",
    model_df["cell_id"].nunique(),
)

print(
    "Model-ready weeks:",
    model_df["week"].nunique(),
)

print(
    "Period:",
    model_df["week"].min(),
    "→",
    model_df["week"].max(),
)

Original rows: 822680
Model-ready rows: 175411
Retention rate: 21.32%
Model-ready cells: 2615
Model-ready weeks: 270
Period: 2019-01-13 00:00:00 → 2024-12-22 00:00:00


In [23]:
example_cell = model_df["cell_id"].iloc[0]

gap_check = ch4_df.loc[
    (ch4_df["cell_id"] == example_cell)
    & (
        ch4_df["week"].between(
            "2022-07-10",
            "2022-09-11",
        )
    ),
    [
        "week",
        "CH4",
        "ch4_prev_1w",
        "ch4_prev_2w",
        "ch4_mean_last_3w",
        "target_next_week",
    ],
]

gap_check

,week,CH4,ch4_prev_1w,ch4_prev_2w,ch4_mean_last_3w,target_next_week
184,2022-07-10,1889.895386,1883.860107,NaN,NaN,1892.848633
185,2022-07-17,1892.848633,1889.895386,1883.860107,1888.868042,NaN
186,2022-07-24,NaN,1892.848633,1889.895386,NaN,NaN
187,2022-07-31,NaN,NaN,1892.848633,NaN,NaN
188,2022-08-07,NaN,NaN,NaN,NaN,NaN
189,2022-08-14,NaN,NaN,NaN,NaN,1898.372314
190,2022-08-21,1898.372314,NaN,NaN,NaN,1913.169800
191,2022-08-28,1913.169800,1898.372314,NaN,NaN,1883.055786
192,2022-09-04,1883.055786,1913.169800,1898.372314,1898.199300,1897.853271
193,2022-09-11,1897.853271,1883.055786,1913.169800,1898.026286,1895.870972


In [ ]:
number_of_candidate_cells = int(
    candidate_ds["candidate_mask"].sum().values
)

2620

In [29]:
assert len(model_df) > 0

assert model_df[required_columns].notna().all().all()

assert model_df["cell_id"].nunique() <= number_of_candidate_cells

assert (
    model_df["target_week"] - model_df["week"]
).dt.days.eq(7).all()

print("Final dataset checks passed.")

Final dataset checks passed.


In [30]:
columns_to_save = [
    "week",
    "target_week",
    "cell_id",
    "x",
    "y",
    "CH4",
    "ch4_prev_1w",
    "ch4_prev_2w",
    "ch4_change_1w",
    "ch4_mean_last_3w",
    "ch4_std_last_3w",
    "season_sin",
    "season_cos",
    "target_next_week",
]

model_df_to_save = (
    model_df[columns_to_save]
    .sort_values(["week", "cell_id"])
    .reset_index(drop=True)
)

In [31]:
processed_data_dir = (
    project_root
    / "data"
    / "processed"
)

processed_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

multiyear_features_path = (
    processed_data_dir
    / "methane_forecasting_features_2019_2024.parquet"
)

model_df_to_save.to_parquet(
    multiyear_features_path,
    index=False,
)

print("Saved to:", multiyear_features_path)

Saved to: c:\Users\Pietro\OneDrive\Desktop\Po-Valley-Methane-Forecasting\data\processed\methane_forecasting_features_2019_2024.parquet


In [32]:
saved_model_df = pd.read_parquet(
    multiyear_features_path
)

print("Saved shape:", saved_model_df.shape)
print("Saved period:")
print(
    saved_model_df["week"].min(),
    "→",
    saved_model_df["week"].max(),
)

print("Saved cells:", saved_model_df["cell_id"].nunique())
print("Saved weeks:", saved_model_df["week"].nunique())

assert saved_model_df.shape == model_df_to_save.shape
assert list(saved_model_df.columns) == list(
    model_df_to_save.columns
)
assert saved_model_df[required_columns].notna().all().all()

print("Saved Parquet dataset successfully verified.")

Saved shape: (175411, 14)
Saved period:
2019-01-13 00:00:00 → 2024-12-22 00:00:00
Saved cells: 2615
Saved weeks: 270
Saved Parquet dataset successfully verified.


In [33]:
candidate_ds.close()